# Assignment3: Volume Rendering, Neural Radiance Fields, Neural Surfaces [lingyun3@andrew.cmu.edu]

## 0. Transmittance Calculation
<div align="center">
    <img src="0/A-0.jpg" width="50%">
</div>

## 1. Differentiable Volume Rendering
### 1.3 Ray Sampling
<div align="center">
    <img src="1.3/_rays.png" width="50%">
    <img src="1.3/_xy_grid.png" width="50%">
</div>

### 1.4 Point Sampling
<div align="center">
    <img src="1.4/_points.png" width="50%">
</div>

### 1.5 Volume Rendering
The depth map is rendered by copying the depth value to the three channels.

<div align="center">
    <img src="1.5/_depth.png" width="50%">
    <img src="1.5/part_1.gif" width="50%">
</div>

## 2. Optimizing a Basic Implicit Volume
### 2.2 Loss and training
```txt
Box center: (0.25007468461990356, 0.25053104758262634, -0.00036301324144005775)

Box side lengths: (2.004605293273926, 1.5033038854599, 1.5031601190567017)
```

### 2.3 Visualization
The first four pictures are before training, and the last four pictures are after training.

The GIF of different camera views is in the last row.

<div align="center">
    <img src="2.3/part_2_before_training_0.png" width="25%">
    <img src="2.3/part_2_before_training_1.png" width="25%">
    <img src="2.3/part_2_before_training_2.png" width="25%">
    <img src="2.3/part_2_before_training_3.png" width="25%">
    <img src="2.3/part_2_after_training_0.png" width="25%">
    <img src="2.3/part_2_after_training_1.png" width="25%">
    <img src="2.3/part_2_after_training_2.png" width="25%">
    <img src="2.3/part_2_after_training_3.png" width="25%">
    <img src="2.3/part_2.gif" width="50%">
</div>

## 3. Optimizing a Neural Radiance Field (NeRF)
Positional encoding is applied.

I first find that the output of density head will be all zero after several iterations when noise standard is zero (which is the case in default configurations). So I raise it to be 1 (according to the official implementation of NeRF) and get the following results:

<div align="center">
    <img src="3/lego_part_3.gif" width="50%">
</div>

## 4. NeRF Extras
### 4.1 View Dependence
Position encoding is applied for both view and 3D position.
The model architecture is borrowed from NeRF as follows:
<div align="center">
    <img src="nerf.png" width="50%">
</div>

The lego scene adopts the NeRF model architecure and the material scene adopts the default architecture for high resolution.

<div align="center">
    <img src="4/lego_full.gif" width="50%">
    <img src="4/materials_part_3.gif" width="50%">
</div>

## 5. Sphere Tracing
Sphere tracing is achieved by using a `for` loop that maintains a mask for non-intersecting points and updating the distance according to the sphere-tracing algorithm at each iteration step. The ground-truth distance is acquired by inferring the implicit funcition. The `for` loop terminates when `max_iter` is achieved or all points have intersections.

<div align="center">
    <img src="5/part_5.gif" width="50%">
</div>

## 6. Optimizing a Neural SDF
The first generated gif is trained with eikonal loss (w=0.02) and the second is trained without eikonal loss (w=0.0). The observation is that training with eikonal loss takes longer to converge (8000 epochs w.r.t 3000 epochs) but the generated surface is more smooth given the regularization.

<div align="center">
    <img src="6/part_6_input.gif" width="50%">
    <img src="6/part_6_with_eikonal.gif" width="50%">
    <img src="6/part_6_wo_eikonal.gif" width="50%">
</div>

## 7. VolSDF
- understanding the parameters for `distance_to_density` function: 

    - α: Controls the maximum density of the object. It sets the overall "opacity" or "solidity" of the volume. A higher α means the object appears more solid/opaque in the interior.

    - β: Controls how sharply the density transitions near the surface boundary. Small β creates a sharp, sudden transition from solid to empty space.

- How does high `beta` bias your learned SDF? What about low `beta`?

High `beta` creates a "fuzzy" surface with gradual density changes. The SDF will tend to learn smoother, more averaged surfaces since the density transition is spread out over a larger region.

Low `beta` forces very sharp surface boundaries. The SDF will try to learn precise surface locations since the density changes abruptly.

- Would an SDF be easier to train with volume rendering and low `beta` or high `beta`? Why?

High `beta` will be easier since gradual transitions provide smoother gradients for optimization.

- Would you be more likely to learn an accurate surface with high `beta` or low `beta`? Why?

Low β is more likely to learn an accurate surface since it forces the model to be very precise about surface locations (the high `beta` allows for cheating by accepting fuzzy surfaces)

<div align="center">
    <img src="7/part_7_geometry.gif" width="50%">
    <img src="7/part_7.gif" width="50%">
</div>

- Hyperparameter settings

I use the default settings and the results look good to me :)

## 8. Neural Surface Extras
### 8.1 Render a Large Scene with Sphere Tracing
I define a `ComplexScene.yaml` configuration that includes a list of primitives (box, torus, sphere) with relevant center and radius parameters in addition to scale and orientation so that the scene can be evenly scattered with multiple primitives. The number of primitives is arbitrary, and I set it to be 20 according to the requirements.

I define a `CompositeSDF` class in `implicit.py` which takes in the `ComplexScene` `cfg` and initiates each primitive. For the `forward` process, it returns the `min` value for all primitives to acquire the union of these primitives.

The results are rendered as follows with the same feature and different resolutions (1024 and 128) as follows:

<div align="center">
    <img src="8.1/part_8.1_1024x.gif" width="50%">
    <img src="8.1/part_8.gif" width="50%">
</div>

### 8.2 Fewer Training Views
THe training views are limited to 20 by applying a slice to `train_idx`. The `val_idx` and `test_idx` are kept the same. The results are as follows: 

<div align="center">
    <img src="8.2/part_8.2.gif" width="50%">
</div>

### 8.3
I borrow the `dist_to_density` implementation proposed in NeuS with different values of `s` (s=10, 30, 90). The results are as follows: 
<div align="center">
    <img src="8.3/part_7_s=10.gif" width="50%">
    <img src="8.3/part_7_s=30.gif" width="50%">
    <img src="8.3/part_7_s=90.gif" width="50%">
</div>

- Analysis: 

The s value controls both the value bound and the deviation of density distribution. Higher s value indicates more denser scene and smaller deviation, which is reflected in the above visualization. s=90 yields brighter scene with more features rendered for the foreground pixels and more consistent in the color distribution.
